# 07 — Log Analysis & SIEM Basics

**Author:** Bency (Benjamin Adjei)  
**Course:** M.S. Cybersecurity — Security Operations  
**Tools:** Python 3, re, pandas, json

---

## Objectives
- Understand common log formats (syslog, auth.log, web access logs)
- Parse and extract fields from raw log lines using regex
- Detect indicators of compromise (failed logins, brute force, 404 sweeps)
- Understand how SIEM tools correlate events
- Generate an alert summary report

## 1. Setup & Imports

In [ ]:
import re
import json
from collections import Counter, defaultdict
from datetime import datetime

## 2. Log Format Overview

| Log Type | Location (Linux) | Key Fields |
|----------|-----------------|------------|
| Auth/SSH | `/var/log/auth.log` | timestamp, user, src IP, success/fail |
| Syslog | `/var/log/syslog` | timestamp, host, process, message |
| Web Access | `/var/log/apache2/access.log` | IP, method, URL, status code, size |
| Firewall | `/var/log/ufw.log` | src/dst IP, port, action (ALLOW/BLOCK) |

### What is a SIEM?
A **Security Information and Event Management (SIEM)** system:
- **Collects** logs from multiple sources
- **Normalizes** them into a common format
- **Correlates** events across sources
- **Alerts** on patterns that match threat rules

Examples: **Splunk**, **Microsoft Sentinel**, **IBM QRadar**, **Elastic SIEM**

## 3. Sample Log Data

In [ ]:
# Simulated /var/log/auth.log entries
auth_logs = """
Jan 15 08:23:01 server sshd[1234]: Failed password for root from 203.0.113.5 port 22 ssh2
Jan 15 08:23:04 server sshd[1234]: Failed password for root from 203.0.113.5 port 22 ssh2
Jan 15 08:23:07 server sshd[1234]: Failed password for root from 203.0.113.5 port 22 ssh2
Jan 15 08:23:10 server sshd[1234]: Failed password for root from 203.0.113.5 port 22 ssh2
Jan 15 08:23:13 server sshd[1234]: Failed password for root from 203.0.113.5 port 22 ssh2
Jan 15 08:23:16 server sshd[1234]: Accepted password for bency from 192.168.1.10 port 52341 ssh2
Jan 15 09:01:00 server sshd[2345]: Failed password for admin from 198.51.100.7 port 22 ssh2
Jan 15 09:01:02 server sshd[2345]: Failed password for admin from 198.51.100.7 port 22 ssh2
Jan 15 09:01:04 server sshd[2345]: Failed password for admin from 198.51.100.7 port 22 ssh2
Jan 15 10:15:33 server sshd[3456]: Accepted publickey for bency from 192.168.1.10 port 54001 ssh2
""".strip().splitlines()

# Simulated Apache access log entries
web_logs = """
203.0.113.5 - - [15/Jan/2025:08:30:01 +0000] "GET /admin HTTP/1.1" 404 512
203.0.113.5 - - [15/Jan/2025:08:30:02 +0000] "GET /wp-login.php HTTP/1.1" 404 512
203.0.113.5 - - [15/Jan/2025:08:30:03 +0000] "GET /phpmyadmin HTTP/1.1" 404 512
203.0.113.5 - - [15/Jan/2025:08:30:04 +0000] "GET /.env HTTP/1.1" 404 512
192.168.1.10 - - [15/Jan/2025:09:00:00 +0000] "GET /index.html HTTP/1.1" 200 2048
10.0.0.5 - - [15/Jan/2025:09:05:00 +0000] "POST /login HTTP/1.1" 200 512
198.51.100.7 - - [15/Jan/2025:10:00:00 +0000] "GET /etc/passwd HTTP/1.1" 400 128
""".strip().splitlines()

print(f'Auth log lines : {len(auth_logs)}')
print(f'Web log lines  : {len(web_logs)}')

## 4. Parsing Auth Logs

In [ ]:
# Regex to parse auth.log SSH lines
AUTH_PATTERN = re.compile(
    r'(?P<timestamp>\w+ \d+ \d+:\d+:\d+) \S+ sshd\[\d+\]: '
    r'(?P<result>Failed|Accepted) \S+ for (?P<user>\S+) from (?P<ip>[\d.]+)'
)

parsed_auth = []
for line in auth_logs:
    m = AUTH_PATTERN.search(line)
    if m:
        parsed_auth.append(m.groupdict())

print(f'Parsed {len(parsed_auth)} auth events:\n')
for e in parsed_auth:
    print(f"  [{e['result']:8}] user={e['user']:10} ip={e['ip']:15} time={e['timestamp']}")

## 5. Parsing Web Access Logs

In [ ]:
# Combined Log Format regex
WEB_PATTERN = re.compile(
    r'(?P<ip>[\d.]+) .+ \[(?P<time>[^\]]+)\] '
    r'"(?P<method>\S+) (?P<url>\S+) HTTP/\S+" (?P<status>\d{3}) (?P<size>\d+)'
)

parsed_web = []
for line in web_logs:
    m = WEB_PATTERN.search(line)
    if m:
        parsed_web.append(m.groupdict())

print(f'Parsed {len(parsed_web)} web events:\n')
for e in parsed_web:
    print(f"  {e['ip']:15} {e['method']:4} {e['url']:25} -> {e['status']}")

## 6. Threat Detection Rules

### Rule 1 — SSH Brute Force
Trigger: **≥ 5 failed SSH logins** from the same IP

In [ ]:
BRUTE_FORCE_THRESHOLD = 5
alerts = []

# Count failed logins per IP
failed_by_ip = Counter(
    e['ip'] for e in parsed_auth if e['result'] == 'Failed'
)

for ip, count in failed_by_ip.items():
    if count >= BRUTE_FORCE_THRESHOLD:
        alert = f'[CRITICAL] SSH Brute Force: {ip} — {count} failed attempts'
        alerts.append({'severity': 'CRITICAL', 'rule': 'SSH_BRUTE_FORCE', 'ip': ip, 'count': count})
        print(alert)

print(f'\nFailed login counts by IP: {dict(failed_by_ip)}')

### Rule 2 — Web Reconnaissance (404 Sweep)

In [ ]:
RECON_THRESHOLD = 3

# Count 404s per IP
notfound_by_ip = Counter(
    e['ip'] for e in parsed_web if e['status'] == '404'
)

for ip, count in notfound_by_ip.items():
    if count >= RECON_THRESHOLD:
        alerts.append({'severity': 'HIGH', 'rule': 'WEB_RECON_404_SWEEP', 'ip': ip, 'count': count})
        print(f'[HIGH] Web Recon (404 sweep): {ip} — {count} not-found requests')

# Check for sensitive URL patterns
SENSITIVE = re.compile(r'(\.env|passwd|wp-login|phpmyadmin|admin)', re.IGNORECASE)
for e in parsed_web:
    if SENSITIVE.search(e['url']):
        alerts.append({'severity': 'HIGH', 'rule': 'SENSITIVE_URL_ACCESS', 'ip': e['ip'], 'url': e['url']})
        print(f"[HIGH] Sensitive URL probe: {e['ip']} requested {e['url']}")

### Rule 3 — Cross-Source Correlation
An IP that appears in **both** failed SSH attempts **and** web reconnaissance is high priority.

In [ ]:
ssh_attacker_ips  = {e['ip'] for e in parsed_auth if e['result'] == 'Failed'}
web_attacker_ips  = {e['ip'] for e in parsed_web  if e['status'] == '404'}

correlated = ssh_attacker_ips & web_attacker_ips

if correlated:
    for ip in correlated:
        alerts.append({'severity': 'CRITICAL', 'rule': 'CORRELATED_MULTI_VECTOR', 'ip': ip})
        print(f'[CRITICAL] Multi-vector attack correlated: {ip} active on SSH + Web')
else:
    print('No cross-source correlation found.')

## 7. Alert Summary Report

In [ ]:
report = {
    'report_time': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'total_auth_events': len(parsed_auth),
    'total_web_events' : len(parsed_web),
    'total_alerts'     : len(alerts),
    'alerts_by_severity': dict(Counter(a['severity'] for a in alerts)),
    'alerts': alerts
}

print(json.dumps(report, indent=2))

## 8. How SIEM Tools Map to This Workflow

| Step in this notebook | SIEM equivalent |
|----------------------|----------------|
| Reading log files | Log ingestion / data connectors |
| Regex parsing | Field extraction / normalization |
| Counting failures | Aggregation queries (SPL / KQL) |
| Threshold rules | Alert rules / detection rules |
| Cross-source correlation | Correlation rules / Fusion alerts |
| JSON report | Incident dashboard / ticket creation |

### Splunk SPL equivalent for brute force:
```spl
index=auth sourcetype=sshd "Failed password"
| stats count by src_ip
| where count >= 5
| sort -count
```

### Microsoft Sentinel KQL equivalent:
```kql
SecurityEvent
| where EventID == 4625
| summarize FailCount = count() by IpAddress
| where FailCount >= 5
| order by FailCount desc
```

## 9. Key Takeaways

- Log parsing starts with understanding the format and writing the right regex
- Threshold-based rules catch brute force and scanning patterns
- **Correlation across log sources** is the most powerful SIEM capability
- Python is an excellent rapid-prototyping tool before deploying to a full SIEM
- Always tune thresholds to reduce false positives in production

---
*Next: 08 — Python Security Automation Scripts*